# Agent Evaluation

NOTE:: I can not evaluate my agent since with ollama I tried only 2 models and I do not get intermediate_trajectories and this means that the agent does not use the search tool

NOTE: 
either I use manuall agent or I use kestra, or openAI
TODO:  TRY loading another model from ollama and check the results

source: https://www.youtube.com/watch?v=lbEj3Waxs1U&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv&index=45

For RAG, we used the A->Q->A' setup:

A = original answer in the FAQ
Q = generated question from this answer
A' = answer produced by our RAG system

* For agents, we use the same setup. A' comes from an agent instead of a fixed RAG pipeline.


* We also save the trajectory. Here, the trajectory means only the tool calls the agent made before producing the final answer.

1. Given the ground_truth data set that is generated by RAGBase,
2. for each record,  get the A' (answer from agent that has memory),
3. save the A', the original answer A, the question, and document to question_with_agent_and_human_answers_data.csv
4. create an judge_agent to judge (score, and resoning),
5. use search_text since we find the optimal parameters

# NOTE 1:::: the agent with memory is more expensive that agent without memory
# NOTE 2::::  I will use agent without memory

In [1]:
import sys
import os
import json
import pandas as pd
from rich import print
from minsearch import Index
import re

parent_directory = os.path.dirname(os.getcwd())
sys.path.append(parent_directory)

from ingest import (load_faq_data,
                    build_index,
                    text_search
)
# load the documents, fillter llm_zoocamp ONLY since the ground_truth data is generated from the llm_zoomcamp ONLY for simplicity
documents = load_faq_data()
documents_llm = [doc for doc in documents if doc['course'] == "llm-zoomcamp"]
print(f'The number of documents = {len(documents_llm)}')

# get the created ground-truth data
df_ground_truth = pd.read_csv('./data/ground_truth_FAQ_data.csv')
ground_truth = df_ground_truth.to_dict(orient="records")
# convert the df to dictionary

# note some answers do not have 5 questions
print(f'The number of ground_truth = {len(ground_truth)}')

# create search index and fit with document (note when I fit, I do not need to use the optimal parameters, only for search I need to use the optimal parameters)
search_index = build_index(documents_llm)

# # Bind your fitted index to the search function using a lambda
# bound_text_search = lambda query: text_search(query=query, search_index=search_index)

The number of documents = 113

The number of ground_truth = 510

In [2]:
rec = ground_truth[0]
rec

{'question': "Okay, so I’m really confused about Office Hours. Where do I actually find the Zoom link? It's not publicly available, right?",
 'document': '489dd1c9d9'}

# Create a lookup table:

In [3]:
doc_idx = {}

for doc in documents:
    doc_idx[doc["id"]] = doc

In [4]:
original_document = doc_idx[rec['document']]
original_document

{'id': '489dd1c9d9',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'What is the video/zoom link to the stream for the “Office Hours” or live/workshop sessions?',
 'answer': 'The zoom link is only published to instructors/presenters/TAs.\n\nStudents participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the [announcements channel on Telegram and Slack](https://t.me/dezoomcamp) before it begins. You can also watch live on the DataTalksClub [YouTube Channel](https://www.youtube.com/c/DataTalksClub).\n\nDon’t post questions in chat as they may be missed if the room is very active.'}

# Agent with a Memory

In [8]:
from smolagents import CodeAgent, LiteLLMModel, Tool
from langchain_core.messages import HumanMessage, AIMessage


class FAQSearchTool(Tool):
    name = "text_search_tuned"
    description = (
        "Searches the LLM Zoomcamp FAQ database using optimized keyword weights. "
        "Use this for any course registration, schedules, or prerequisite questions."
    )
    inputs = {
        "query": {
            "type": "string",
            "description": "The simple search keywords or queries to pass into the FAQ database."
        }
    }

    output_type = "string"

    def __init__(self, search_index, **kwargs):
        super().__init__(**kwargs)
        self.search_index = search_index

    def forward(self, query: str) -> str:

        boost_dict = {
            "question": 3.0,
            "section": 0.5,
            "answer": 10.0,
        }

        filter_dict = {
            "course": "llm-zoomcamp"
        }
        search_results = self.search_index.search(
            query,
            num_results=5,
            boost_dict=boost_dict,
            filter_dict=filter_dict
        )
        lines = []

        for doc in search_results:
            lines.append(f"Section: {doc['section']}")
            lines.append(f"Q: {doc['question']}")
            lines.append(f"A: {doc['answer']}\n")
        return "\n".join(lines).strip() if lines else "No matching FAQ entries found."


class FAQAgent:
    def __init__(self, search_index):
        # Memory
        self.chat_history = []
    
        # Model
        self.model = LiteLLMModel(
            model_id="ollama_chat/qwen2.5:3b-instruct",
            api_base="http://127.0.0.1:11434"
        )
        # Tool
        self.faq_tool = FAQSearchTool(
            search_index=search_index
        )
        # Agent
        self.agent = CodeAgent(
            model=self.model,
            tools=[self.faq_tool],
            max_steps=5,
            verbosity_level=0
        )

    def get_history(self):

        if not self.chat_history:
            return "No previous conversation."

        history = []
        for message in self.chat_history:
            if isinstance(message, HumanMessage):
                history.append(
                    f"User: {message.content}"
                )
            elif isinstance(message, AIMessage):
                history.append(
                    f"Assistant: {message.content}"
                )
        return "\n".join(history)
    
    def clear_memory(self):
        """I want to clear memory after each query to reduce the cost"""
        self.chat_history = []

    def update_memory(self, question, answer):
        self.chat_history.append(
            HumanMessage(content=question)
        )
        self.chat_history.append(
            AIMessage(content=answer)
        )
        # keep last 10 messages
        self.chat_history = self.chat_history[-10:]

    def run(self, question):
        history = self.get_history()
        task_prompt = f"""
You are an FAQ assistant for LLM Zoomcamp.
Conversation history:
{history}

You have one tool:
text_search_tuned(query)

Follow these steps:
1. Call text_search_tuned with the user's question.
2. Read the returned FAQ information.
3. Answer ONLY using the returned information.
4. Do not copy the search results.
5. Keep the answer concise.

IMPORTANT:
Your final output MUST be inside a code block exactly like this:

<code>
final_answer("your answer here")
</code>

Never write final_answer outside the code block.

User question:
{question}
"""
        result = self.agent.run(
            task_prompt,
            return_full_result=True
        )
        answer = result.output

        # Update memory
        self.update_memory(
            question,
            answer
        )
        return result
    

def get_query_cost_by_agent_with_memeory(agent_result):
    usage_agent_result = {
        "input_tokens": 0,
        "output_tokens": 0,
        "total_tokens": 0,
    }
    for step in agent_result.steps:
        if isinstance(step, dict):
            step_usage = step.get("token_usage")
            if step_usage:
                for key in usage_agent_result:
                    usage_agent_result[key] += step_usage.get(key, 0)
    return(usage_agent_result)



def get_tool_call_by_agent_with_memeory(agent_result):
    tool_calls = []

    for step in agent_result.steps:
        if not isinstance(step, dict):
            continue
        code = step.get("code_action")

        if not code:
            continue
        matches = re.findall(
            r'(\w+)\((.*?)\)',
            code,
            re.DOTALL
        )
        for name, args in matches:
            if name != "print":
                tool_calls.append(
                    {
                        "tool": name,
                        "arguments": args
                    }
                )
    return tool_calls



faq_agent_with_memory = FAQAgent(search_index)


question = rec['question']
# I will clear the history to create agent withut memory since I want to reduce the cost
faq_agent_with_memory.clear_memory()
search_index_agent_result = faq_agent_with_memory.run(question)

print("Final answer:")
print(search_index_agent_result.output)


Final answer:

The Zoom link for the Office Hours is only published to instructors/presenters/TAs. Students can participate via 
YouTube Live and submit questions using Slido.

In [10]:
usage_agent_result = get_query_cost_by_agent_with_memeory(search_index_agent_result)
usage_agent_result

{'input_tokens': 5461, 'output_tokens': 79, 'total_tokens': 5540}

In [11]:
# get tool_call (agent trajectories)
tool_calls = get_tool_call_by_agent_with_memeory(search_index_agent_result)
tool_calls

[{'tool': 'text_search_tuned', 'arguments': 'query="Office Hours Zoom link"'},
 {'tool': 'final_answer',
  'arguments': '"The Zoom link for the Office Hours is only published to instructors/presenters/TAs. Students can participate via YouTube Live and submit questions using Slido."'}]

### For a Document, report its id, question, original and agent answer, cost and tool_calls



In [12]:
def generate_agent_answer(record_from_ground_truth):
    original_document = doc_idx[record_from_ground_truth['document']]

    # I will create  an instance each time since I want agent without a amemory
    faq_agent_with_memory = FAQAgent(search_index)
    faq_agent_with_memory.clear_memory() # to be sure that its memory cleared although I am not si-ure about that 100 percent
    
    question = original_document['question']
    search_index_agent_result = faq_agent_with_memory.run(question)
    answer_agent = search_index_agent_result.output
    tool_calls = get_tool_call_by_agent_with_memeory(search_index_agent_result)
    usage_agent_result = get_query_cost_by_agent_with_memeory(search_index_agent_result)

    record_from_ground_truth_to_report = {
        "question": question,
        "answer_agent": answer_agent,
        "answer_orig": original_document["answer"],
        "tool_calls": tool_calls,
        "cost": usage_agent_result,
        "document": original_document['id'],
    }
    return record_from_ground_truth_to_report

generate_agent_answer(ground_truth[0])

{'question': 'What is the video/zoom link to the stream for the “Office Hours” or live/workshop sessions?',
 'answer_agent': "The zoom link for 'Office Hours' or live/workshop sessions is only published to instructors/presenters/TAs. Students can participate via YouTube Live and submit questions using Slido (link pinned in chat when live). The video URL should be posted on the announcements channel on Telegram and Slack, and DataTalks.Club YouTube Channel.",
 'answer_orig': 'The zoom link is only published to instructors/presenters/TAs.\n\nStudents participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the [announcements channel on Telegram and Slack](https://t.me/dezoomcamp) before it begins. You can also watch live on the DataTalksClub [YouTube Channel](https://www.youtube.com/c/DataTalksClub).\n\nDon’t post questions in chat as they may be missed if the room is very active.',
 'tool_calls': [{'tool': 'text

# Run in Parallel

In [ ]:
from evaluation_utils import  run_in_parallel
agent_answers= run_in_parallel(
    func=generate_agent_answer,
    items=ground_truth,
)


Processing:   1%|          | 6/510 [00:14<16:53,  2.01s/it]

Code parsing failed on line 7 due to: SyntaxError: unterminated string literal (detected at line 7) (<unknown>, 
line 7)
4. Submit answers through the course platform before the deadline.")                                               
^

Code parsing failed on line 3 due to: SyntaxError: unterminated string literal (detected at line 3) (<unknown>, 
line 3)
"For Module 1 in the 2026 cohort, you can start here: ^

Code parsing failed on line 7 due to: SyntaxError: unterminated string literal (detected at line 7) (<unknown>, 
line 7)
4. Submit answers through the course platform before the deadline.")                                               
^

Code parsing failed on line 1 due to: SyntaxError: unterminated string literal (detected at line 1) (<unknown>, 
line 1)
"Start with the LLM Zoomcamp docs, general Zoomcamp logistics docs, and LLM Zoomcamp GitHub repository. The typical
workflow includes watching lesson videos, working through notebooks, reading homework instructions, and submitting 
answers on time. ^

...

Reached max steps.

Code parsing failed on line 1 due to: SyntaxError: unterminated string literal (detected at line 1) (<unknown>, 
line 1)
**Codespace won't connect / stuck on 'Finishing up':**                                                   ^

Code parsing failed on line 1 due to: SyntaxError: unexpected character after line continuation character 
(<unknown>, line 1)
python_interpreter(text="'''\n"**Codespace won\'t connect / stuck on \\"Finishing up\\":**\\n- Go to 
[github.com/codespaces](https://github.com/codespaces), stop the codespace, and start it again.\\n- If it still 
won\\\'t connect, open it in the **browser** instead of desktop VS Code, or try a different browser 
(Edge/Chrome/Brave).\\n- As a last resort, delete the codespace and create a new one.""\n\n"**My setup is all 
gone:**\\n\\n- The repo in `/workspaces` persists across stop/start, but a **rebuild or a brand-new codespace** 
starts from a clean image, and system/global installs outside your project don\'t always survive. Reinstalling is 
quick with `uv` (`uv sync` / `uv add ...`).\\n- Commit and push your work often — uncommitted changes survive a 
stop/start but are lost if you delete or recreate the codespace."\n'''\n")
                                                ^

Reached max steps.

Reached max steps.

In [ ]:
df_agent = pd.DataFrame(agent_answers)

In [ ]:
df_agent.to_csv("./data/agent_answers.csv", index=False)